# Proyecto 3er Parcial — Análisis Quejas Habitacionales (Databricks)

**Materia:** Tópicos de Big Data · **Periodo:** agosto-diciembre de 2026 · **Tecnologías:** Databricks Free Edition, PySpark, Spark SQL, Delta Lake, Unity Catalog y Tableau Public.

**Equipo:** Issabela Marian Romero Aguayo, Deborah Villicaña Prieto


## Etapa 1 · Ingesta y preparación

Importación de librerías

In [0]:
import os
import math
import time
import requests
from pyspark.sql import functions as F


In [0]:
URL_BASE = "https://data.cityofnewyork.us/resource"
APP_TOKEN = None          # opcional: "tu_app_token"
HEADERS = {"X-App-Token": APP_TOKEN} if APP_TOKEN else {}

INICIO = "2025-01-01T00:00:00"
FIN    = "2026-01-01T00:00:00"   # límite exclusivo: incluye todo el 31/12/2025

# Fuente 1: 311 Service Requests (solo quejas de vivienda atendidas por HPD)
ID_311 = "erm2-nwe9"
COLUMNAS_311 = [
    "unique_key", "created_date", "closed_date", "agency", "complaint_type",
    "descriptor", "location_type", "incident_zip", "borough", "status",
    "resolution_action_updated_date", "community_board", "bbl",
    "latitude", "longitude", "open_data_channel_type"
]
WHERE_311 = f"agency = 'HPD' AND created_date >= '{INICIO}' AND created_date < '{FIN}'"

# Fuente 2: HPD Housing Maintenance Code Violations
ID_VIOL = "wvxf-dwi5"
COLUMNAS_VIOL = [
    "violationid", "buildingid", "boroid", "boro", "zip", "class",
    "inspectiondate", "approveddate", "novissueddate", "novtype",
    "currentstatus", "currentstatusdate", "violationstatus", "rentimpairing",
    "communityboard", "councildistrict", "bbl", "nta", "latitude", "longitude"
]
WHERE_VIOL = f"novissueddate >= '{INICIO}' AND novissueddate < '{FIN}'"

# Catálogo de referencia: población por distrito
ID_POB = "xywu-7bv9"

Prueba la conexión con una consulta pequeña (`count(*)`). Un código HTTP 200 indica que Databricks puede llegar a la API. 

In [0]:
#def contar_registros(dataset_id, where):
#    """Cuenta en el servidor cuántas filas cumplen el filtro."""
#    r = requests.get(
#        f"{URL_BASE}/{dataset_id}.json",
#        params={"$select": "count(*) AS total", "$where": where},
#        headers=HEADERS,
#        timeout=60
#    )
#    r.raise_for_status()
#    return int(r.json()[0]["total"])


#print("Quejas 311 (HPD, 2025):", f"{contar_registros(ID_311, WHERE_311):,}")
#print("Violaciones HPD (2025):", f"{contar_registros(ID_VIOL, WHERE_VIOL):,}")

#NO FUNCIONO

Crea el catálogo, los esquemas `stage`, `clean` y `datamart`, y el volumen donde se guardarán los archivos de origen.

In [0]:
CATALOGO = "nyc_vivienda"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
for esquema in ["stage", "clean", "datamart"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{esquema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.stage.archivos_origen")

RUTA_VOLUMEN = f"/Volumes/{CATALOGO}/stage/archivos_origen"
CARPETA_311  = f"{RUTA_VOLUMEN}/quejas_311_hpd_2025"
CARPETA_VIOL = f"{RUTA_VOLUMEN}/violaciones_hpd_2025"
CARPETA_POB  = f"{RUTA_VOLUMEN}/poblacion_distrito"

spark.sql(f"SHOW SCHEMAS IN {CATALOGO}").show(truncate=False)

In [0]:
# Genera las URL para descarga manual (solo si hace falta el plan B)
from urllib.parse import urlencode

def url_api(dataset_id, columnas, where, limite=1_000_000):
    params = {"$select": ",".join(columnas), "$where": where, "$limit": limite}
    return f"{URL_BASE}/{dataset_id}.csv?" + urlencode(params)

print(url_api(ID_311, COLUMNAS_311, WHERE_311), "\n")
print(url_api(ID_VIOL, COLUMNAS_VIOL, WHERE_VIOL), "\n")
print(url_api(ID_POB, ["age_group", "borough", "_2020"], "age_group = 'Total Population'"))

Crea las subcarpetas dentro del volumen con esta celda nueva

In [0]:
for carpeta in [CARPETA_311, CARPETA_VIOL, CARPETA_POB]:
    os.makedirs(carpeta, exist_ok=True)
    print("Carpeta lista:", carpeta)

In [0]:
for carpeta in [CARPETA_311, CARPETA_VIOL, CARPETA_POB]:
    for archivo in os.listdir(carpeta):
        tamano_mb = os.path.getsize(f"{carpeta}/{archivo}") / (1024 ** 2)
        print(f"{carpeta.split('/')[-1]}/{archivo}: {tamano_mb:.1f} MB")

## Etapa 2 · Subir los archivos al volumen

In [0]:
def leer_csv(carpeta):
    return (
        spark.read
        .option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .csv(f"{carpeta}/*.csv")
    )

df_311_raw  = leer_csv(CARPETA_311)
df_viol_raw = leer_csv(CARPETA_VIOL)
df_pob_raw  = leer_csv(CARPETA_POB)

# Referencia: conteos que devolvía la API para 2025 (pueden variar un poco por actualizaciones)
print(f"Quejas 311 leídas:  {df_311_raw.count():,} (referencia ≈ 774,590)")
print(f"Violaciones leídas: {df_viol_raw.count():,} (referencia ≈ 868,018)")

In [0]:
import os

ruta_volumen = (
    "/Volumes/nyc_vivienda/stage/"
    "archivos_origen/"
)

archivos = [
    "quejas_311_hpd_2025/erm2-nwe9.csv",
    "violaciones_hpd_2025/wvxf-dwi5.csv",
    "poblacion_distrito/xywu-7bv9.csv"
]

for archivo in archivos:
    tamano_mb = os.path.getsize(ruta_volumen + archivo) / (1024 ** 2)
    print(f"{archivo}: {tamano_mb:.2f} MB")


tamano_kb = os.path.getsize(ruta_volumen + "poblacion_distrito/xywu-7bv9.csv") / 1024
print(f"Población: {tamano_kb:.2f} KB")

## Etapa 3 · Leer los archivos con Spark

Lee las quejas 311 como DataFrame; "header" interpreta encabezados, y "inferSchema" deduce tipos.

In [0]:
ruta_quejas = (
    "/Volumes/nyc_vivienda/stage/"
    "archivos_origen/quejas_311_hpd_2025/erm2-nwe9.csv"
)

df_quejas = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ruta_quejas)
)

df_quejas.printSchema()

Lee las violaciones del código de mantenimiento de HPD con las mismas opciones.

In [0]:
ruta_violaciones = (
    "/Volumes/nyc_vivienda/stage/"
    "archivos_origen/violaciones_hpd_2025/wvxf-dwi5.csv"
)

df_violaciones = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ruta_violaciones)
)

df_violaciones.printSchema()

Lee el catálogo de población por distrito. Se usa solo como referencia para calcular incidencia por habitante.

In [0]:
ruta_poblacion = (
    "/Volumes/nyc_vivienda/stage/"
    "archivos_origen/poblacion_distrito/xywu-7bv9.csv"
)

df_poblacion = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ruta_poblacion)
)

df_poblacion.printSchema()

## Etapa 4 · Ver una muestra y contar los registros

##QUEJAS 311
Inspecciona una muestra pequeña de las quejas. `limit` evita traer todos los registros a la interfaz.

In [0]:
display(df_quejas.limit(10))

Cuenta filas y columnas de las quejas. **Referencia:** alrededor de 774,590 quejas de HPD en 2025.

In [0]:
num_filas = df_quejas.count()
num_columnas = len(df_quejas.columns)

print(f"Número de quejas: {num_filas:,}")
print(f"Número de columnas: {num_columnas}")

##VIOLACIONES HPD
Inspecciona una muestra de las violaciones y cuenta sus registros. **Referencia:** alrededor de 868,018 violaciones emitidas en 2025.

In [0]:
display(df_violaciones.limit(10))

num_filas = df_violaciones.count()
num_columnas = len(df_violaciones.columns)

print(f"Número de violaciones: {num_filas:,}")
print(f"Número de columnas: {num_columnas}")

Muestra el catálogo completo de población; es pequeño, así que no se necesita `limit`.

In [0]:
display(df_poblacion)

## Etapa 5 · Diagnóstico de calidad

5.1. Revisar valores nulos y vacíos

In [0]:
from pyspark.sql import functions as F

def diagnostico_calidad(df, nombre):
    total = df.count()

    print(f"\n{'='*55}")
    print(f"DIAGNÓSTICO: {nombre}")
    print(f"{'='*55}")
    print(f"Filas: {total:,}")
    print(f"Columnas: {len(df.columns)}")

    resultados = []

    for columna in df.columns:
        nulos = df.filter(
            F.col(columna).isNull() |
            (F.trim(F.col(columna).cast("string")) == "")
        ).count()

        resultados.append((
            columna,
            nulos,
            round((nulos / total) * 100, 2) if total else 0
        ))

    reporte = spark.createDataFrame(
        resultados,
        ["columna", "valores_nulos", "porcentaje_nulos"]
    )

    display(reporte.orderBy(F.desc("porcentaje_nulos")))

diagnostico_calidad(df_quejas, "NYC 311")
diagnostico_calidad(df_violaciones, "Violaciones HPD")
diagnostico_calidad(df_poblacion, "Población")

5.2. Revisar duplicados

In [0]:
print("DUPLICADOS EN NYC 311")

print(
    "Registros duplicados por unique_key:",
    df_quejas.count() - df_quejas.dropDuplicates(["unique_key"]).count()
)

print("\nDUPLICADOS EN HPD")

print(
    "Registros duplicados por violationid:",
    df_violaciones.count() -
    df_violaciones.dropDuplicates(["violationid"]).count()
)

5.3. Revisar fechas y categorías

In [0]:
print("TIPOS DE QUEJA")
display(
    df_quejas.groupBy("complaint_type")
    .count()
    .orderBy(F.desc("count"))
)

print("ESTADOS DE LAS QUEJAS")
display(
    df_quejas.groupBy("status")
    .count()
    .orderBy(F.desc("count"))
)

print("DISTRIBUCIÓN TERRITORIAL 311")
display(
    df_quejas.groupBy("borough")
    .count()
    .orderBy(F.desc("count"))
)

print("CLASES DE VIOLACIONES HPD")
display(
    df_violaciones.groupBy("class")
    .count()
    .orderBy("class")
)

## Etapa 6 · Reglas de limpieza

### Reglas de limpieza aplicadas

**Fuente NYC 311**
- Se conserva únicamente la agencia HPD, correspondiente a quejas atendidas por el Departamento de Preservación y Desarrollo de Vivienda.
- Se consideran registros creados entre el 1 de enero y el 31 de diciembre de 2025.
- Se eliminan registros sin identificador único.
- Se eliminan duplicados utilizando `unique_key`.
- Se normalizan las fechas de creación y cierre.
- Los registros sin fecha de cierre se conservan como quejas pendientes.
- Las fechas de cierre anteriores a la creación se consideran inválidas y no se utilizan para calcular duración.
- Los tipos de queja y distritos sin información se clasifican como no identificados.

**Fuente HPD**
- Se consideran violaciones emitidas durante 2025, utilizando `novissueddate`.
- Se eliminan registros sin identificador de violación.
- Se eliminan duplicados utilizando `violationid`.
- Se normalizan las fechas y los nombres de los distritos.
- Se conservan las clases de violación para analizar su distribución.

**Catálogo de población**
- Se utiliza la categoría Total Population.
- Se convierte la población a formato numérico.
- Se normalizan los nombres de los distritos para facilitar la integración territorial.

**Criterios generales**
- No se eliminan automáticamente valores extremos.
- No se confunden quejas 311 con violaciones HPD: representan eventos distintos.
- Los registros sin cierre no participan en el cálculo de duración, pero se contabilizan como pendientes.

## Etapa 7 · Limpieza con PySpark

7.1. Limpiar NYC 311

In [0]:
from pyspark.sql import functions as F

# Eliminar registros sin identificador y duplicados
df_311_limpio = (
    df_quejas
    .filter(F.col("unique_key").isNotNull())
    .dropDuplicates(["unique_key"])
)

# Convertir fechas
df_311_limpio = (
    df_311_limpio
    .withColumn(
        "fecha_creacion",
        F.to_timestamp("created_date")
    )
    .withColumn(
        "fecha_cierre_original",
        F.to_timestamp("closed_date")
    )
)

# Conservar únicamente fechas de creación de 2025
df_311_limpio = df_311_limpio.filter(
    (F.col("fecha_creacion") >= F.to_timestamp(F.lit("2025-01-01"))) &
    (F.col("fecha_creacion") < F.to_timestamp(F.lit("2026-01-01")))
)

# Detectar fechas de cierre válidas
df_311_limpio = (
    df_311_limpio
    .withColumn(
        "fecha_cierre",
        F.when(
            F.col("fecha_cierre_original") >= F.col("fecha_creacion"),
            F.col("fecha_cierre_original")
        )
    )
    .withColumn(
        "tipo_queja",
        F.coalesce(F.col("complaint_type"), F.lit("No identificado"))
    )
    .withColumn(
        "distrito",
        F.upper(F.trim(F.col("borough")))
    )
    .withColumn(
        "estado_analisis",
        F.when(F.col("fecha_cierre").isNull(), "Pendiente")
         .otherwise("Cerrada")
    )
    .withColumn(
        "dias_cierre",
        F.when(
            F.col("fecha_cierre").isNotNull(),
            (F.unix_timestamp("fecha_cierre") -
             F.unix_timestamp("fecha_creacion")) / 86400
        )
    )
)

print("Registros NYC 311 después de limpieza:", df_311_limpio.count())

display(df_311_limpio.select(
    "unique_key",
    "tipo_queja",
    "distrito",
    "fecha_creacion",
    "fecha_cierre",
    "estado_analisis",
    "dias_cierre"
).limit(10))

7.2. Limpiar violaciones HPD

In [0]:
df_hpd_limpio = (
    df_violaciones
    .filter(F.col("violationid").isNotNull())
    .dropDuplicates(["violationid"])
    .withColumn(
        "fecha_emision",
        F.to_timestamp("novissueddate")
    )
    .withColumn(
        "distrito",
        F.upper(F.trim(F.col("boro")))
    )
    .withColumn(
        "clase_violacion",
        F.coalesce(F.col("class"), F.lit("No identificada"))
    )
)

# Conservar violaciones emitidas durante 2025
df_hpd_limpio = df_hpd_limpio.filter(
    (F.col("fecha_emision") >= F.to_timestamp(F.lit("2025-01-01"))) &
    (F.col("fecha_emision") < F.to_timestamp(F.lit("2026-01-01")))
)

print("Violaciones HPD después de limpieza:", df_hpd_limpio.count())

display(df_hpd_limpio.select(
    "violationid",
    "clase_violacion",
    "distrito",
    "fecha_emision",
    "currentstatus"
).limit(10))

7.3. Limpiar población

In [0]:
df_poblacion_limpia = (
    df_poblacion
    .filter(F.col("age_group") == "Total Population")
    .select(
        F.upper(F.trim(F.col("borough"))).alias("distrito"),
        F.col("_2020").cast("long").alias("poblacion")
    )
    .filter(F.col("poblacion").isNotNull())
    .dropDuplicates(["distrito"])
)

display(df_poblacion_limpia)

7.4. Validación después de la limpieza

In [0]:
print("RESUMEN DE LIMPIEZA")

print("NYC 311:", f"{df_311_limpio.count():,}")
print("HPD:", f"{df_hpd_limpio.count():,}")
print("Población:", f"{df_poblacion_limpia.count():,}")

print("\nESTADO DE LAS QUEJAS")
display(
    df_311_limpio.groupBy("estado_analisis")
    .count()
)

print("\nDISTRIBUCIÓN DE QUEJAS POR DISTRITO")
display(
    df_311_limpio.groupBy("distrito")
    .count()
    .orderBy(F.desc("count"))
)

## Etapa 8 · Persistencia Delta

8.1. Guardar las tablas limpias

In [0]:
# Guardar NYC 311
(
    df_311_limpio.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("nyc_vivienda.clean.quejas_311")
)

# Guardar violaciones HPD
(
    df_hpd_limpio.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("nyc_vivienda.clean.violaciones_hpd")
)

# Guardar población
(
    df_poblacion_limpia.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("nyc_vivienda.clean.poblacion_distrito")
)

print("Tablas Delta guardadas correctamente.")

8.2. Comprobar que las tablas existen

In [0]:
spark.sql("SHOW TABLES IN nyc_vivienda.clean").show(truncate=False)

In [0]:
df_311_delta = spark.table("nyc_vivienda.clean.quejas_311")
df_hpd_delta = spark.table("nyc_vivienda.clean.violaciones_hpd")
df_pob_delta = spark.table("nyc_vivienda.clean.poblacion_distrito")

print("Quejas Delta:", df_311_delta.count())
print("Violaciones Delta:", df_hpd_delta.count())
print("Población Delta:", df_pob_delta.count())

## Etapa 9 · Datamart

9.1. Datamart 1: volumen mensual de quejas

In [0]:
dm_quejas_mensuales = (
    df_311_delta
    .withColumn(
        "mes",
        F.date_format("fecha_creacion", "yyyy-MM")
    )
    .groupBy("mes")
    .agg(
        F.count("*").alias("total_quejas"),
        F.countDistinct("unique_key").alias("quejas_unicas")
    )
    .orderBy("mes")
)

display(dm_quejas_mensuales)

9.2. Datamart 2: tiempos de cierre

In [0]:
dm_tiempos_cierre = (
    df_311_delta
    .groupBy("tipo_queja")
    .agg(
        F.count("*").alias("total_quejas"),

        F.sum(
            F.when(F.col("estado_analisis") == "Cerrada", 1)
             .otherwise(0)
        ).alias("quejas_cerradas"),

        F.sum(
            F.when(F.col("estado_analisis") == "Pendiente", 1)
             .otherwise(0)
        ).alias("quejas_pendientes"),

        F.round(
            F.avg("dias_cierre"), 2
        ).alias("promedio_dias_cierre"),

        F.round(
            F.percentile_approx("dias_cierre", 0.5), 2
        ).alias("mediana_dias_cierre"),

        F.round(
            F.percentile_approx("dias_cierre", 0.9), 2
        ).alias("percentil_90_dias")
    )
    .orderBy(F.desc("promedio_dias_cierre"))
)

display(dm_tiempos_cierre)

9.3. Datamart 3: violaciones por distrito

In [0]:
dm_violaciones_distrito = (
    df_hpd_delta
    .withColumn(
        "distrito_consejo",
        F.concat_ws(
            " - ",
            F.col("distrito"),
            F.coalesce(
                F.col("councildistrict").cast("string"),
                F.lit("No identificado")
            )
        )
    )
    .groupBy("distrito", "distrito_consejo")
    .agg(
        F.countDistinct("violationid").alias("total_violaciones")
    )
    .orderBy(F.desc("total_violaciones"))
)

display(dm_violaciones_distrito)

9.4. Datamart 4: comparación territorial de incidencia

In [0]:
# Quejas 311 por borough
quejas_territorio = (
    df_311_delta
    .groupBy("distrito")
    .agg(
        F.countDistinct("unique_key").alias("total_quejas")
    )
)

# Violaciones HPD por borough
violaciones_territorio = (
    df_hpd_delta
    .groupBy("distrito")
    .agg(
        F.countDistinct("violationid").alias("total_violaciones")
    )
)

# Unir ambos conteos y población
dm_incidencia_territorial = (
    df_pob_delta
    .join(quejas_territorio, on="distrito", how="left")
    .join(violaciones_territorio, on="distrito", how="left")
    .fillna({
        "total_quejas": 0,
        "total_violaciones": 0
    })
    .withColumn(
        "quejas_por_10000_habitantes",
        F.round(
            F.col("total_quejas") /
            F.col("poblacion") * 10000, 2
        )
    )
    .withColumn(
        "violaciones_por_10000_habitantes",
        F.round(
            F.col("total_violaciones") /
            F.col("poblacion") * 10000, 2
        )
    )
    .orderBy(F.desc("quejas_por_10000_habitantes"))
)

display(dm_incidencia_territorial)

9.5. Guardar los cuatro Datamarts como tablas Delta

In [0]:
datamarts = {
    "quejas_mensuales": dm_quejas_mensuales,
    "tiempos_cierre": dm_tiempos_cierre,
    "violaciones_distrito": dm_violaciones_distrito,
    "incidencia_territorial": dm_incidencia_territorial
}

for nombre, df in datamarts.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"nyc_vivienda.datamart.{nombre}")
    )

print("Los cuatro Datamarts se guardaron correctamente.")

In [0]:
spark.sql("SHOW TABLES IN nyc_vivienda.datamart").show(truncate=False)

## Etapa 10 · Respuesta a la pregunta

## Etapa 11 · Exportación

##Etapa 12 · Dashboard y presentación

Construcción del dashboard en Tableau Public

**Exporta únicamente los resultados agregados**, no los 10.8 millones de viajes. Desde cada `display(...)` en Databricks utiliza la descarga CSV del resultado (si la interfaz no lo permite, exporta los DataFrames agregados a un volumen).

1. **Evolución diaria:** `demanda_diaria.csv`. Columnas: `fecha_viaje` como fecha continua; Filas: `SUM(total_viajes)`; Marcas: línea.
2. **Demanda por día:** misma fuente diaria. Columnas: `dia_semana` discreto; Filas: `AVG(total_viajes)`; Marcas: barras. Crea alias de domingo a sábado.
3. **Mapa de calor:** `demanda_horaria.csv`. Columnas: `hora` discreta; Filas: `dia_semana` discreto; Marcas: cuadrado; Color: promedio de viajes por día y hora. Para este trimestre, divide por 12 los martes y por 13 los demás días, o utiliza el número de ocurrencias obtenido desde `demanda_diaria`.
4. **Top 10 zonas:** `demanda_por_zona.csv`. Filas: `zona`; Columnas: `SUM(total_viajes)`; barras horizontales; filtro Top 10 por suma de viajes.

**Dashboard:** crea un dashboard nuevo en el **mismo libro**; coloca la evolución diaria arriba, barras semanales y ranking al centro, y mapa de calor abajo. Añade título, fuente, periodo y notas metodológicas. Las fuentes agregadas son diferentes: los filtros entre hojas no funcionan automáticamente sin relaciones/acciones configuradas.

### Entregables
- Notebook ejecutado con resultados y respuestas a las preguntas.
- Captura de las validaciones principales y explicación de las reglas de limpieza.
- Cuatro hojas y un dashboard publicado en Tableau Public.
- Conclusiones: tres hallazgos sustentados por datos y dos limitaciones del análisis.

### Reflexión final
¿Por qué la ingesta se realizó en Databricks y no en la laptop? ¿Qué diferencia hay entre una vista temporal y una tabla Delta? ¿Por qué no eliminamos automáticamente tarifas negativas ni valores superiores al P99? ¿Por qué usamos `AVG(total_viajes)` para comparar días de la semana?
